# Авито Антибот: оценка cookie по событиям

Откройте ноутбук из корня проекта и выполните **Run All**. Нужен Python 3.10 и пакеты из `requirements.txt`. Результат записывается в `submission.csv`.


## Идея

Каждой cookie соответствует суточное окно. Сначала отсекаем события вне `[window_start_ts, window_end_ts)`, затем сворачиваем последовательность событий в одну строку признаков на cookie. Признаки описывают интенсивность и ритм действий, глубину поиска, разнообразие объявлений, контактные действия, движение курсора и User-Agent. Бустинг учится отличать размеченный трафик известных сервисов сбора данных от трафика, отнесённого к человеческому.

`cookie_id` сохраняется только как ключ; сам идентификатор не передаётся модели. Метаданные и события доступны до конца окна.


In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

from metric import precision_at_recall


SEED = 42
EVENT_TYPES = [
    "search_results_view", "item_view", "photo_swipe", "seller_page_view",
    "contact_phone_show", "contact_chat_open", "contact_message_sent",
    "favorite_add", "login", "captcha_shown",
]

## 1. Построение признаков

Число действий само по себе неоднозначно: активный человек тоже может открыть много объявлений. Поэтому считаем также доли типов событий, интервалы между ними, повторы, глубину страниц и простые переходы между действиями. Пропуски в полях событий заменяются нулём уже после агрегации.


In [2]:
def make_features(meta, events):
    e = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id")
    e = e.loc[(e.event_ts >= e.window_start_ts) & (e.event_ts < e.window_end_ts)].copy()
    e = e.sort_values(["cookie_id", "event_ts"], kind="stable")
    g = e.groupby("cookie_id", sort=False)
    f = pd.DataFrame(index=meta.cookie_id)
    f.index.name = "cookie_id"
    f["cookie_age_days"] = ((meta.cookie_created_at - meta.window_start_ts) / pd.Timedelta(days=1)).to_numpy() * -1
    f["created_in_window"] = (meta.cookie_created_at >= meta.window_start_ts).to_numpy().astype(int)
    f["n_events"] = g.size()
    f["active_seconds"] = (g.event_ts.max() - g.event_ts.min()).dt.total_seconds()
    f["first_event_hour"] = ((g.event_ts.min() - g.window_start_ts.first()) / pd.Timedelta(hours=1))
    f["last_event_hour"] = ((g.event_ts.max() - g.window_start_ts.first()) / pd.Timedelta(hours=1))
    f["n_active_hours"] = g.event_ts.apply(lambda s: s.dt.hour.nunique())
    f["n_items"] = g.item_id.nunique()
    f["n_categories"] = g.item_category.nunique()
    f["n_locations"] = g.item_location.nunique()
    f["n_queries"] = g.search_query.nunique()
    f["n_agents"] = g.user_agent.nunique()
    f["n_platforms"] = g.platform.nunique()
    f["max_category_share"] = e.groupby(["cookie_id", "item_category"]).size().groupby(level=0).max() / g.item_category.count()
    f["max_item_views"] = e.loc[e.event_name.eq("item_view")].groupby(["cookie_id", "item_id"]).size().groupby(level=0).max()

    counts = pd.crosstab(e.cookie_id, e.event_name).reindex(columns=EVENT_TYPES, fill_value=0)
    f = f.join(counts.add_prefix("n_"))
    for name in EVENT_TYPES:
        f["share_" + name] = f["n_" + name] / f.n_events
    f["items_per_view"] = f.n_items / f.n_item_view.clip(lower=1)
    f["queries_per_search"] = f.n_queries / f.n_search_results_view.clip(lower=1)
    f["views_per_search"] = f.n_item_view / f.n_search_results_view.clip(lower=1)
    f["contacts_per_view"] = (f.n_contact_phone_show + f.n_contact_chat_open + f.n_contact_message_sent) / f.n_item_view.clip(lower=1)

    e["gap"] = g.event_ts.diff().dt.total_seconds()
    gaps = e.groupby("cookie_id").gap
    for stat in ["mean", "median", "std", "min", "max"]:
        f["gap_" + stat] = getattr(gaps, stat)()
    for seconds in [1, 5, 30, 300, 3600]:
        f[f"gap_le_{seconds}"] = e.gap.le(seconds).groupby(e.cookie_id).mean()
    f["gap_cv"] = f.gap_std / f.gap_mean.clip(lower=1)
    f["gap_p10"] = gaps.quantile(0.1)
    f["gap_p90"] = gaps.quantile(0.9)
    f["n_sessions"] = e.gap.gt(1800).groupby(e.cookie_id).sum() + 1
    hour_counts = e.groupby(["cookie_id", e.event_ts.dt.floor("h")]).size()
    f["max_events_hour"] = hour_counts.groupby(level=0).max()
    f["events_hour_std"] = hour_counts.groupby(level=0).std()
    for event_type in ["item_view", "search_results_view"]:
        subset = e.loc[e.event_name.eq(event_type), ["cookie_id", "event_ts"]].copy()
        subset["same_gap"] = subset.groupby("cookie_id").event_ts.diff().dt.total_seconds()
        same_gaps = subset.groupby("cookie_id").same_gap
        for stat in ["median", "std"]:
            f[f"{event_type}_gap_{stat}"] = getattr(same_gaps, stat)()
        f[f"{event_type}_gap_le_30"] = subset.same_gap.le(30).groupby(subset.cookie_id).mean()

    e["platform_norm"] = e.platform.str.lower().replace({"desktop": "web", "iphone": "ios"})
    for platform in ["web", "android", "ios"]:
        f["platform_" + platform] = e.platform_norm.eq(platform).groupby(e.cookie_id).mean()
    ua = e.user_agent.fillna("").str.lower()
    for name, pattern in {
        "headless": "headless", "python": "python", "bot": "bot", "curl": "curl",
        "selenium": "selenium", "playwright": "playwright", "firefox": "firefox",
        "yabrowser": "yabrowser", "mobile": "mobile", "linux": "linux",
    }.items():
        f["ua_" + name] = ua.str.contains(pattern, regex=False).groupby(e.cookie_id).mean()

    for column in ["search_page", "pointer_x", "pointer_y"]:
        s = pd.to_numeric(e[column], errors="coerce")
        f[column + "_present"] = s.notna().groupby(e.cookie_id).mean()
        for stat in ["mean", "std", "max"]:
            f[column + "_" + stat] = getattr(s.groupby(e.cookie_id), stat)()
        if column == "search_page":
            for page in [2, 5, 10]:
                f[f"search_page_ge_{page}"] = s.ge(page).groupby(e.cookie_id).sum()
        else:
            f[column + "_nunique"] = s.groupby(e.cookie_id).nunique()
    f["max_query_repeats"] = e.groupby(["cookie_id", "search_query"]).size().groupby(level=0).max()
    f["max_search_page_per_query"] = e.groupby(["cookie_id", "search_query"]).search_page.max().groupby(level=0).max()

    e["prev_event"] = g.event_name.shift()
    for before, after in [
        ("search_results_view", "item_view"), ("item_view", "item_view"),
        ("item_view", "contact_phone_show"), ("item_view", "photo_swipe"),
        ("captcha_shown", "item_view"),
    ]:
        f[f"transition_{before}_{after}"] = (
            e.prev_event.eq(before) & e.event_name.eq(after)
        ).groupby(e.cookie_id).sum()

    return f.replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")

## 2. Данные и таблица cookie × признаки

Обучающие и тестовые cookie обрабатываются одной функцией, чтобы набор колонок и порядок вычисления совпадали. Разметка теста нигде не используется.


In [3]:
dates = ["cookie_created_at", "window_start_ts", "window_end_ts"]

train = pd.read_csv("data/train.csv", parse_dates=dates)

test = pd.read_csv("data/test.csv", parse_dates=dates)

events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

all_meta = pd.concat([train, test], ignore_index=True)

x = make_features(all_meta, events)

x_train = x.loc[train.cookie_id]

x_test = x.loc[test.cookie_id]

y = train.target.to_numpy()
print(f"train: {train.shape}, test: {test.shape}, events: {events.shape}, features: {x.shape}")

train: (11091, 5), test: (4909, 4), events: (328905, 14), features: (16000, 98)


## 3. Проверка по будущим датам

Тест относится к датам после обучающей выборки. Поэтому для проверки обучаем модель на ранних днях и измеряем качество на следующих днях. Используем официальную `precision_at_recall`: она выбирает максимальную точность среди порогов с полнотой не ниже 70%. Порог в файл ответа не записывается.


In [4]:
model = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.045, max_leaf_nodes=15,
    l2_regularization=3, random_state=SEED,
)

for start, end in [("2026-04-14", "2026-04-17"), ("2026-04-17", "2026-04-20")]:
    fit = train.window_start_ts.lt(start).to_numpy()
    valid = train.window_start_ts.ge(start).to_numpy() & train.window_start_ts.lt(end).to_numpy()
    model.fit(x_train.iloc[fit], y[fit])
    pred = model.predict_proba(x_train.iloc[valid])[:, 1]
    print(f"{start}..{end}: P@R>=0.7={precision_at_recall(y[valid], pred):.4f}, "
          f"PR-AUC={average_precision_score(y[valid], pred):.4f}, "
          f"ROC-AUC={roc_auc_score(y[valid], pred):.4f}")

2026-04-14..2026-04-17: P@R>=0.7=0.7366, PR-AUC=0.7642, ROC-AUC=0.9306
2026-04-17..2026-04-20: P@R>=0.7=0.7847, PR-AUC=0.7818, ROC-AUC=0.9339


## 4. Обучение на всём train и файл ответа

После проверки обучаем ту же модель на всех размеченных cookie. Для каждой cookie из test получаем score от 0 до 1 и проверяем формат перед сохранением.


In [5]:
model.fit(x_train, y)

submission = pd.DataFrame({"cookie_id": test.cookie_id, "score": model.predict_proba(x_test)[:, 1]})

assert submission.cookie_id.is_unique and submission.cookie_id.equals(test.cookie_id)

assert submission.score.notna().all() and submission.score.between(0, 1).all()

submission.to_csv("submission.csv", index=False)

print(f"Saved submission.csv with {len(submission)} rows ({len(x.columns)} features)")

Saved submission.csv with 4909 rows (98 features)
